# 05b One record patch: the ISE arm of SecFid's Table 1 gets its comparison, and every derived file is rebuilt

Design log.
- What was missing. The SecFid record from the six-paper pilot (notebook 03) has an evaluation for each of the four defences re-measured in its Table 1 (page 6), but a comparison against the undefended row for only three of them: ASIDE, Meta-SecAlign and DefensiveTokens. The ISE arm (`ev_ise`) has its three printed values in the record and no comparison. Notebook 05 therefore computed the suppression cost for three defences and not for ISE, although ISE's row sits in the same table under the same conditions.
- The patch. One comparison `c_ise` is added to `reports/records/secfid_arxiv2026.yaml`: `ev_ise` against `ev_base`, purpose W, the same six conditions as `c_aside` (layout unmatched by design of a W comparison, everything else matched). The headline claim `k1`, which speaks of the defences of Table 1 together, lists `c_ise` among its supporting comparisons. Nothing else in the record changes. The file is loaded and written back with the same settings as the record builder, after a check that the round trip is byte-identical.
- Checks on the patched record: the frozen schema, every printed value and anchor on its page in the PDF, the amended coding rules, and the corpus-wide checks.
- Derived files. The extraction tables, the decisive values, the analysis tables, `findings_05.md` and `decision_05.txt` are all computed from the records, so they are regenerated here from the same code paths: `extract.build_tables` and `extract.decisive_results` (notebook 04), and the output stage of notebook 05, which is moved into `analysis.run` so that later notebooks regenerate the same files from the records without copying cells. The decisive-value rows of SecFid are recomputed from its PDF; the rows of the other 123 reports are unchanged and kept. `decision_04.txt` is left as written; its counts predate this patch by one comparison and three decisive values, and `decision_05b.txt` says so.
- The two new printed values the findings use (ISE's Exec. 14.0 and Ign. 20.3, Table 1, page 6) were read by eye against the page before this notebook was delivered, as were the eight already in use.
- CPU only; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. Gate: notebook 05 is committed and the record is as committed

In [ ]:
NEED = ["reports/decision_05.txt", "reports/findings_05.md", "reports/analysis/scs.csv", "reports/extraction/decisive_values_04.csv"]
missing = [p for p in NEED if not Path(p).exists()]
untracked = not missing and subprocess.run(["git", "ls-files", "--error-unmatch", *NEED], capture_output=True, text=True).returncode != 0
dirty = subprocess.run(["git", "status", "--porcelain", "--", *NEED, "reports/records", "src", "schema"], capture_output=True, text=True).stdout.strip()
if missing or untracked or dirty:
    raise SystemExit(f"STOP: run and commit notebook 05 first (missing: {missing}; not committed: {untracked}; changed: {dirty[:200]})")
C04 = subprocess.run(["git", "log", "-1", "--format=%h", "--", "reports/decision_04.txt"], capture_output=True, text=True).stdout.strip()
C05 = subprocess.run(["git", "log", "-1", "--format=%h", "--", "reports/decision_05.txt"], capture_output=True, text=True).stdout.strip()
print("notebook 04 in", C04, "| notebook 05 in", C05)

## Stage 2. The modules: `analysis.run` (the output stage of notebook 05) and the tests

In [ ]:
%%writefile src/analysis.py
"""The analyses of plan v2 A7 over the re-coded records: the map, signal dependence, layout compatibility, coverage
(by paper, by configuration, jointly, original and by anyone), the metric crosswalk, claim support, supported
orderings, stress-test coverage and the suppression cost per security point.

Notebook 05 writes this module. Nothing here calls a model or the network. Every row keeps the record ids it was
counted from, so every number in the findings can be traced to its records."""
from __future__ import annotations

import re
from collections import Counter, defaultdict

QUANTITIES = ["R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe"]
AXIS1 = ["chat_roles", "delimiters_markers", "encodings", "learned_token_features", "fixed_embedding_transform", "position",
         "internal_intervention", "no_signal"]
AXIS2 = ["prompting", "soft_tokens_frozen", "sft", "preference_optimisation", "distillation", "adversarial_rl", "fixed_inference_rule"]
LAYOUTS = ["user_turn", "separate_role", "system_context", "agent_tool_result"]
PROMPT_OR_RULE = {"prompting", "fixed_inference_rule"}
STRESS = {"adaptive", "optimisation", "rl", "human"}
SIG_ORDER = ["measured", "measured_other_checkpoint", "stated", "baseline_only", "not_reported", "not_applicable"]
GRADE_ORDER = ["A", "B", "C", "U"]

# method families: other papers' runs of a method are linked to it by name (first rule that matches wins)
FAMILY_RULES = [
    (r"meta[\s-]*sec\s*align|metasec\b|secalign\s*\+\+", "Meta-SecAlign"),
    (r"\bsec\s*align\b", "SecAlign"),
    (r"\bstruq\b", "StruQ"),
    (r"defensive\s*tokens?", "DefensiveTokens"),
    (r"spotlight", "Spotlighting"),
    (r"sandwich|repeat\w*\s+(the\s+)?(user('s)?\s+)?(prompt|instruction)|\brup\b", "Sandwich (instruction repeated after the data)"),
    (r"instruction\s+hierarchy", "Instruction Hierarchy"),
    (r"^reminder\b|explicit reminder", "Reminder prompt"),
    (r"instructional|instruction\s+defen[cs]e", "Instructional prompt"),
    (r"\bise\b", "ISE"), (r"\baside\b", "ASIDE"), (r"jatmo", "Jatmo"), (r"\bdrip\b", "DRIP"), (r"secopd", "SecOPD"),
    (r"verih", "VerIH"), (r"\breta\b", "RETA"), (r"agentalign", "AgentAlign"), (r"reasalign", "ReasAlign"),
    (r"polymorphic\s+prompt|\bppa\b", "Polymorphic Prompt Assembling"), (r"\bfath\b", "FATH"),
]


def stem(name: str) -> str:
    """The method's name without trailing parentheses: 'ACT (activation consistency training)' -> 'ACT'."""
    s = name.strip()
    while True:
        t = re.sub(r"\s*\([^()]*\)\s*$", "", s).strip()
        if t == s or not t:
            return s
        s = t


NOT_A_FAMILY = r"\s\+\s|\binto\b|hardened|guardrail with|white-box defence"   # a new method built on a named one is its own


def family(name: str) -> str | None:
    """The named method family, or None when the name matches no rule (a generic name is never linked across reports)."""
    n = name.lower()
    if re.search(NOT_A_FAMILY, n):
        return None
    for pat, fam in FAMILY_RULES:
        if re.search(pat, n):
            return fam
    return None


def trained(d: dict) -> bool:
    return not set(d["axis2"]) <= PROMPT_OR_RULE


def family_key(d: dict, rid: str) -> str:
    """Records of the same named family and the same kind (trained, or prompting and fixed rules) share a key across
    reports; any other method keeps a key of its own report."""
    fam = family(d["name"])
    kind = "trained" if trained(d) else "prompting or rule"
    return f"{fam} | {kind}" if fam else f"{rid} | {stem(d['name'])} | {kind}"


def _num(v) -> float | None:
    m = re.match(r"\s*([-+]?\d*\.?\d+)", str(v or ""))
    return float(m.group(1)) if m else None


def as_percent(r: dict) -> float | None:
    """A rate on the 0 to 100 scale, or None when the unit is not a rate."""
    x = _num(r.get("value"))
    if x is None:
        return None
    if r.get("unit") == "percent":
        return x
    if r.get("unit") == "fraction":
        return 100 * x
    return None


def _results_by_eval(rec):
    out = defaultdict(list)
    for r in rec.get("results") or []:
        out[r["evaluation_id"]].append(r)
    return out


def _quantities(r: dict) -> list[tuple[str, str]]:
    """(quantity, coverage) pairs a result speaks to; a second quantity in also_quantity counts as a proxy."""
    out = [(r["quantity"], r["coverage"])]
    out += [(q, "proxy") for q in r.get("also_quantity") or [] if q != r["quantity"]]
    return out


# ---------------------------------------------------------------- methods and the map (A7.1)
# reports whose own records are one method under several names (training recipes, schedules, channels, strengths, a
# scale study, or variants the paper presents as one defence); every main record of the report joins one method
MERGE_REPORTS = {"air_arxiv2025": "AIR", "guardrag_uwthesis2026": "GuardRAG with SAGE-C", "gwdpo_arxiv2026": "GW-DPO",
                 "secpo_arxiv2026": "SecPO", "tracer_emnlp2026": "TRACER", "trustmi_arxiv2026": "trust steering",
                 "spotlighting_arxiv2024": "Spotlighting"}

def methods(records: dict) -> list[dict]:
    """A method is a defence a report runs as its own (origin this_report, not a variant or control record), on any
    number of base checkpoints. Its records are the base-checkpoint records of the same name."""
    groups = defaultdict(list)
    for rid, rec in records.items():
        for d in rec.get("defences") or []:
            if d["origin"] == "this_report" and d["isolation"] != "not_applicable":
                groups[(rid, MERGE_REPORTS.get(rid) or stem(d["name"]))].append(d)
    out = []
    for (rid, name), ds in sorted(groups.items()):
        rep = records[rid]["report"]
        a1 = sorted({a for d in ds for a in d["axis1"]}, key=AXIS1.index)
        a2 = sorted({b for d in ds for b in d["axis2"]}, key=AXIS2.index)
        sig = min((d["signal_dependence"] for d in ds), key=SIG_ORDER.index)
        lc = {}
        for lay in LAYOUTS:
            vals = {(d.get("layout_compat") or {}).get(lay, "untested") for d in ds}
            lc[lay] = "supported" if "supported" in vals else ("unsupported" if "unsupported" in vals else "untested")
        out.append({"method_uid": f"{rid}:{re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')}", "report": rid, "name": name,
                    "family": family(name) or "", "year": str(rep_year(rep)), "role": rep["role"], "axis1": ";".join(a1),
                    "axis2": ";".join(a2), "trained": "yes" if any(trained(d) for d in ds) else "no",
                    "base_records": len(ds), "isolation": ds[0]["isolation"], "signal_dependence": sig,
                    "signal_dependence_detail": next((d.get("signal_dependence_detail") for d in ds
                                                      if d["signal_dependence"] == sig and d.get("signal_dependence_detail")), ""),
                    "released_weights": "yes" if any(d.get("released_weights") for d in ds) else "no",
                    **{f"layout_{k}": v for k, v in lc.items()},
                    "defence_uids": ";".join(f"{rid}:{d['id']}" for d in ds)})
    return out


def rep_year(rep: dict) -> int:
    return int(str(rep.get("first_date", "0000"))[:4] or 0)


def map_table(meths: list[dict]) -> list[dict]:
    """Methods per cell of the map (axis 1 by axis 2), by year; a two-cell method counts in each of its cells."""
    years = sorted({m["year"] for m in meths})
    rows = []
    for a in AXIS1:
        for b in AXIS2:
            ms = [m for m in meths if a in m["axis1"].split(";") and b in m["axis2"].split(";")]
            rows.append({"axis1": a, "axis2": b, "methods": len(ms), "reports": len({m["report"] for m in ms}),
                         **{f"methods_{y}": sum(m["year"] == y for m in ms) for y in years},
                         "released_weights": sum(m["released_weights"] == "yes" for m in ms),
                         "method_uids": ";".join(m["method_uid"] for m in ms)})
    return rows


# ---------------------------------------------------------------- coverage (A7.2)
def configuration_coverage(records: dict) -> list[dict]:
    """By evaluated configuration: for every defence record, the best coverage of each quantity in its own report's
    evaluations, and whether one evaluation scores the jobs jointly on the same replies."""
    rows = []
    for rid, rec in sorted(records.items()):
        by_eval = _results_by_eval(rec)
        evs = defaultdict(list)
        for e in rec.get("evaluations") or []:
            evs[e["defence_id"]].append(e)
        for d in rec.get("defences") or []:
            best, joint3, joint12 = {}, [], []
            for e in evs.get(d["id"], []):
                qs = {}
                for r in by_eval.get(e["id"], []):
                    for q, c in _quantities(r):
                        if q in QUANTITIES:
                            qs[q] = "direct" if "direct" in (qs.get(q), c) else "proxy"
                for q, c in qs.items():
                    best[q] = "direct" if "direct" in (best.get(q), c) else "proxy"
                has3 = any(q.startswith("R3") for q in qs)
                if "R3safe" in qs or ("R1" in qs and "R2attack" in qs and has3):
                    joint3.append(e["id"])
                if "R1" in qs and "R2attack" in qs:
                    joint12.append(e["id"])
            rows.append({"defence_uid": f"{rid}:{d['id']}", "report": rid, "name": d["name"], "origin": d["origin"],
                         "family_key": family_key(d, rid), **{q: best.get(q, "none") for q in QUANTITIES},
                         "joint_all_three": "yes" if joint3 else "no", "joint_R1_R2attack": "yes" if joint12 else "no",
                         "joint_evaluations": ";".join(f"{rid}:{x}" for x in joint3)})
    return rows


def family_coverage(records: dict, cfg_rows: list[dict], meths: list[dict]) -> list[dict]:
    """Original measurement against measurement by anyone: per method family proposed in the corpus, which quantities
    its own report covers and which any report in the corpus covers for the same family."""
    by_fam = defaultdict(list)
    for r in cfg_rows:
        by_fam[r["family_key"]].append(r)
    rows = []
    mkey = {m["method_uid"]: family_key({"name": m["name"], "axis2": m["axis2"].split(";")}, m["report"]) for m in meths}
    for fk in sorted(set(mkey.values())):
        if not fk.split(" | ")[0] in {f for _, f in FAMILY_RULES}:
            continue                                    # only named families can be measured by others
        own_reports = {m["report"] for m in meths if mkey[m["method_uid"]] == fk}
        rs = by_fam.get(fk, [])
        own = [r for r in rs if r["report"] in own_reports and r["origin"] == "this_report"]
        other = [r for r in rs if r["report"] not in own_reports]
        row = {"family_key": fk, "proposing_reports": ";".join(sorted(own_reports)),
               "reports_measuring_it_later": ";".join(sorted({r["report"] for r in other}))}
        for q in QUANTITIES:
            row[f"{q}_original"] = "yes" if any(r[q] != "none" for r in own) else "no"
            row[f"{q}_anyone"] = "yes" if any(r[q] != "none" for r in rs) else "no"
        rows.append(row)
    return rows


# ---------------------------------------------------------------- metric crosswalk (A7.2)
BENCH_RULES = [(r"agentdojo|agentdyn", "AgentDojo or AgentDyn"), (r"injecagent", "InjecAgent"), (r"\bsep\b", "SEP"),
               (r"alpaca\s*eval", "AlpacaEval"), (r"alpaca\s*farm", "AlpacaFarm"), (r"bipia", "BIPIA"),
               (r"\bmmlu\b", "MMLU"), (r"\basb\b|agent security bench", "ASB"), (r"wasp", "WASP"),
               (r"open\s*prompt\s*injection|\bopi\b", "OpenPromptInjection"), (r"tasktracker", "TaskTracker"),
               (r"piarena", "PIArena"), (r"cyberseceval", "CyberSecEval"), (r"llmail", "LLMail-Inject"),
               (r"tensor\s*trust", "TensorTrust"), (r"squad", "SQuAD-based"), (r"gsm8k", "GSM8K"),
               (r"natural questions|\bnq\b|hotpot|realtime\s*qa|triviaqa", "Open-domain QA sets"),
               (r"secfid", "SecFid"), (r"mind2web", "Mind2Web")]
JUDGE_RULES = [(r"not (stated|named|described|detailed)|judge not", "not stated"),
               (r"\bhuman\b|raters", "human raters"),
               (r"gpt|claude|gemini|llama|qwen|deepseek|llm|judge|win rate|grader|rubric|evaluator", "model judge"),
               (r"string|substring|literal|starts with|begins with|contains only|keyword|regex|appears in (the )?(response|reply|output)"
                r"|phrase in the response|hacked|uuid|exact|signature|contains the answer|answer changes|target option|\bterm\b|outputs the",
                "string or answer match"),
               (r"checker|checks|check\b|environment|state|tool|function|call|parser|executable|probe|verifier|validator|\bapi\b"
                r"|send_email|exfiltrat|executes", "programmatic checker"),
               (r"accuracy|label|gold|answer key|reference|f1|gleu|rouge|metric|scor|rating", "reference metric"),
               (r"goal|success|rule|protocol|criteri|compliance|elicited|attack|desired|carries out|as ev_", "success rule given, judge not named")]
CANNOT_SEE = {
    "R1": {"string or answer match": "cannot tell an executed injection (E) from a quoted or translated one (Rp); blind to P and S",
           "model judge": "counts E as the judge's prompt defines it; blind to P and S unless scored separately",
           "programmatic checker": "sees the actions or state the attacker wanted, not what the reply says; blind to P and S",
           "reference metric": "counts a wrong answer as attack success whatever caused it; blind to Rp, P and S",
           "not stated": "the rule that counts E is not given",
           "human raters": "counts E as the raters judged it; blind to P and S unless scored separately",
           "success rule given, judge not named": "the success rule is stated, but not whether a string, a model or a person applied it",
           "unclassified": "the record's judge text does not say how E is counted"},
    "R2clean": "no injection in the data: blind to E, Rp and to the user's task under attack",
    "R2attack": "counts P under attack; blind to whether the injection was also executed unless paired with R1 on the same replies",
    "R3ns": "counts S (text dropped); blind to E",
    "R3cp": "counts correct handling of the asked-for content; blind to E unless scored jointly",
    "R3safe": "counts P and not E on the same replies (joint)",
    "general_utility": "not the user's task on the document: blind to E, Rp, P under attack and S",
    "other": "not mapped onto the outcome codes",
}
COUNTS = {"R1": "E (as 1 - Pr[E])", "R2clean": "P on clean data", "R2attack": "P under attack", "R3ns": "S (as 1 - Pr[S])",
          "R3cp": "P restricted to the asked-for content", "R3safe": "P and not E", "general_utility": "none of E, Rp, P, S, F",
          "other": "not mapped"}


def classify(text: str, rules) -> str:
    t = (text or "").lower()
    for pat, lab in rules:
        if re.search(pat, t):
            return lab
    return "the report's own set" if rules is BENCH_RULES else "unclassified"


def crosswalk(records: dict) -> list[dict]:
    groups = defaultdict(lambda: {"results": 0, "reports": set(), "metrics": Counter(), "ids": []})
    for rid, rec in records.items():
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        for r in rec.get("results") or []:
            e = evs.get(r["evaluation_id"], {})
            judge = classify(e.get("judge", ""), JUDGE_RULES)
            key = (classify(e.get("benchmark", ""), BENCH_RULES), r["quantity"], judge)
            g = groups[key]
            g["results"] += 1
            g["reports"].add(rid)
            g["metrics"][r["metric_as_printed"]] += 1
            g["ids"].append(f"{rid}:{r['id']}")
    rows = []
    for (bench, q, judge), g in sorted(groups.items(), key=lambda kv: (-kv[1]["results"], kv[0])):
        cs = CANNOT_SEE[q]
        rows.append({"benchmark_family": bench, "quantity": q, "judge_rule": judge, "outcome_counted": COUNTS[q],
                     "cannot_see": cs.get(judge, cs["unclassified"]) if isinstance(cs, dict) else cs,
                     "results": g["results"], "reports": len(g["reports"]),
                     "metric_names": "; ".join(f"{m} ({n})" for m, n in g["metrics"].most_common(5)),
                     "result_uids": ";".join(g["ids"])})
    return rows


# ---------------------------------------------------------------- claim support and orderings (A7.3)
def claim_support(records: dict, grade) -> list[dict]:
    rows = []
    for rid, rec in sorted(records.items()):
        comps = {c["id"]: c for c in rec.get("comparisons") or []}
        by_eval = _results_by_eval(rec)
        for k in rec.get("claims") or []:
            sup = [comps[c] for c in k.get("supported_by") or [] if c in comps]
            grades = Counter(grade(c) for c in sup)
            best = next((g for g in GRADE_ORDER if grades[g]), "none")
            qs = {q for c in sup for side in ("left", "right") for r in by_eval.get(c[side], []) for q, _ in _quantities(r)}
            jobs = k["jobs"]
            utility = ""
            if "R2" in jobs or "general_utility" in jobs:
                utility = ("R2 results" if qs & {"R2clean", "R2attack"} else
                           "general utility only" if "general_utility" in qs else "no utility result in its comparisons")
            rows.append({"claim_uid": f"{rid}:{k['id']}", "report": rid, "role": rec["report"]["role"], "location": k["location"],
                         "jobs": ";".join(jobs), "supporting_comparisons": len(sup),
                         **{f"grade_{g}": grades[g] for g in GRADE_ORDER},
                         "purposes": ";".join(sorted({c["purpose"] for c in sup})), "best_grade": best,
                         "contradicted_by": len(k.get("contradicted_by") or []), "utility_basis": utility,
                         "quantities_in_support": ";".join(q for q in QUANTITIES + ["general_utility", "other"] if q in qs),
                         "text": k["text"]})
    return rows


def orderings(records: dict, grade) -> list[dict]:
    """Supported orderings: grade-A comparisons between two defended arms inside one report. Cross-report readings
    (purpose X) are listed with their grade; the review states an ordering only inside a grade-A set."""
    rows = []
    for rid, rec in sorted(records.items()):
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        names = {d["id"]: d["name"] for d in rec.get("defences") or []}
        by_eval = _results_by_eval(rec)
        for c in rec.get("comparisons") or []:
            l, r = evs.get(c["left"], {}), evs.get(c["right"], {})
            ld, rd = str(l.get("defence_id", "")), str(r.get("defence_id", ""))
            if c["purpose"] != "X" and (ld.startswith("none:") or rd.startswith("none:")):
                continue
            pairs = [(a, b) for a in by_eval.get(c["left"], []) for b in by_eval.get(c["right"], [])
                     if a["quantity"] == b["quantity"] and a["metric_as_printed"] == b["metric_as_printed"]
                     and a.get("unit") == b.get("unit") and _num(a.get("value")) is not None and _num(b.get("value")) is not None]
            rows.append({"comparison_uid": f"{rid}:{c['id']}", "report": rid, "purpose": c["purpose"], "grade": grade(c),
                         "left_arm": names.get(ld, ld), "right_arm": names.get(rd, rd), "varies": c.get("varies", ""),
                         "paired_results": len(pairs),
                         "result_pairs": ";".join(f"{rid}:{a['id']}|{rid}:{b['id']}" for a, b in pairs)})
    return rows


# ---------------------------------------------------------------- stress tests (A7.4)
def stress_tests(records: dict, meths: list[dict]) -> list[dict]:
    """Per method: the attackers its configurations faced, in its own report and in later reports of the same family."""
    later = defaultdict(list)
    for rid, rec in records.items():
        dk = {d["id"]: family_key(d, rid) for d in rec.get("defences") or []}
        for e in rec.get("evaluations") or []:
            if e["defence_id"] in dk:
                later[dk[e["defence_id"]]].append((rid, e))
    rows = []
    for m in meths:
        fk = family_key({"name": m["name"], "axis2": m["axis2"].split(";")}, m["report"])
        own_ids = {u.split(":", 1)[1] for u in m["defence_uids"].split(";")}
        own = [e for e in records[m["report"]].get("evaluations") or [] if e["defence_id"] in own_ids]
        out = [(rid, e) for rid, e in later.get(fk, []) if rid != m["report"]]
        def kinds(es):
            return sorted({e["attacker_type"] for e in es if e["attacker_type"] in STRESS})
        rows.append({"method_uid": m["method_uid"], "axis1": m["axis1"], "axis2": m["axis2"], "trained": m["trained"], "family": m["family"],
                     "own_static": sum(e["attacker_type"] == "static" for e in own),
                     "own_stress_attackers": ";".join(kinds(own)),
                     "own_white_box": sum(e["attacker_access"] == "white_box" for e in own),
                     "signal_removal_run": "yes" if m["signal_dependence"] in ("measured", "measured_other_checkpoint") else "no",
                     "later_reports": ";".join(sorted({rid for rid, _ in out})),
                     "later_stress_attackers": ";".join(kinds([e for _, e in out])),
                     "later_white_box": sum(e["attacker_access"] == "white_box" for _, e in out),
                     "pushed_hard": "yes" if kinds(own) or kinds([e for _, e in out]) else "no"})
    return rows


# ---------------------------------------------------------------- suppression cost per security point (A7.5)
def scs_pairs(records: dict, grade) -> list[dict]:
    """SCS = (R3ns undefended - R3ns defended) / (R1 defended - R1 undefended), from one grade-A comparison of a
    defended and the undefended evaluation that both carry an R1 and an R3ns rate. Rates on the 0 to 100 scale;
    a rate printed as 'lower is better' (attack success, ignored rate) is turned into R1 or R3ns as 100 minus it."""
    def rate(rs, q):
        for r in rs:
            if q in [x for x, _ in _quantities(r)]:
                v = as_percent(r)
                if v is not None:
                    return (100 - v if r.get("direction") == "lower_better" else v), r
        return None, None
    rows = []
    for rid, rec in sorted(records.items()):
        evs = {e["id"]: e for e in rec.get("evaluations") or []}
        by_eval = _results_by_eval(rec)
        names = {d["id"]: d["name"] for d in rec.get("defences") or []}
        for c in rec.get("comparisons") or []:
            if grade(c) != "A":
                continue
            sides = {s: evs.get(c[s], {}) for s in ("left", "right")}
            und = [s for s, e in sides.items() if str(e.get("defence_id", "")).startswith("none:")]
            if len(und) != 1:
                continue
            u = und[0]
            dsd = "right" if u == "left" else "left"
            r1d, xr1d = rate(by_eval[c[dsd]], "R1")
            r1u, xr1u = rate(by_eval[c[u]], "R1")
            r3d, xr3d = rate(by_eval[c[dsd]], "R3ns")
            r3u, xr3u = rate(by_eval[c[u]], "R3ns")
            if None in (r1d, r1u, r3d, r3u) or r1d == r1u:
                continue
            rows.append({"comparison_uid": f"{rid}:{c['id']}", "report": rid,
                         "defence": names.get(sides[dsd]["defence_id"], sides[dsd]["defence_id"]),
                         "base": sides[u]["defence_id"][5:], "R1_defended": round(r1d, 1), "R1_undefended": round(r1u, 1),
                         "R3ns_defended": round(r3d, 1), "R3ns_undefended": round(r3u, 1),
                         "SCS": round((r3u - r3d) / (r1d - r1u), 2),
                         "result_uids": ";".join(f"{rid}:{x['id']}" for x in (xr1d, xr1u, xr3d, xr3u))})
    return rows


# ---------------------------------------------------------------- the outputs of notebook 05, regenerable from the records
def pct(a, b):
    if not b:
        return f"{a} of 0"
    share = 100 * a / b
    return f"{a} of {b} ({'under 1' if 0 < share < 0.5 else round(share)} in 100)"


def _ids(xs, n=12):
    xs = sorted(xs)
    return ", ".join(xs) if len(xs) <= n else f"{len(xs)} ids in the file"


def run(records: dict, grade, conditions, c04: str, read: dict, decisive: dict, today: str) -> dict:
    """Every table of reports/analysis/, the findings and the decision lines, from the records.
    read: result uid -> the value read by eye against the page, for every printed value the findings use.
    decisive: result uid -> the row of reports/extraction/decisive_values_04.csv (its code check is quoted).
    Stops with a ValueError when a value the findings use was not read against its page or differs."""
    T = {}
    meths = methods(records)
    T["methods"] = meths
    mp = map_table(meths)
    T["map"] = mp
    by_a1 = Counter(a for m in meths for a in m["axis1"].split(";"))
    by_a2 = Counter(b for m in meths for b in m["axis2"].split(";"))
    filled = [r for r in mp if r["methods"]]
    T["signal_dependence"] = [{k: m[k] for k in ("method_uid", "name", "axis1", "axis2", "trained", "signal_dependence",
                                                 "signal_dependence_detail", "defence_uids")} for m in meths]
    T["layout_compat"] = [{k: m[k] for k in ["method_uid", "name", "axis1", "axis2"] + [f"layout_{x}" for x in LAYOUTS] + ["defence_uids"]}
                          for m in meths]
    with_signal = [m for m in meths if m["axis1"] != "no_signal"]
    sig_t = Counter(m["signal_dependence"] for m in with_signal if m["trained"] == "yes")
    sig_p = Counter(m["signal_dependence"] for m in with_signal if m["trained"] == "no")
    paper_rows = [{"report": rid, "role": r["report"]["role"], **{q: r["coverage"][q] for q in QUANTITIES},
                   "where": "; ".join(f"{k}: {v}" for k, v in (r["coverage"].get("where") or {}).items())} for rid, r in sorted(records.items())]
    T["coverage_by_paper"] = paper_rows
    cfg = configuration_coverage(records)
    T["coverage_by_configuration"] = cfg
    fam = family_coverage(records, cfg, meths)
    T["coverage_by_family"] = fam
    cw = crosswalk(records)
    T["crosswalk"] = cw
    cs = claim_support(records, grade)
    T["claim_support"] = cs
    od = orderings(records, grade)
    T["orderings"] = od
    gap_rows = []
    for c in conditions:
        hit = [(rid, k) for rid, r in sorted(records.items()) for k in r.get("comparisons") or [] if k.get(c) == "unreported"]
        gap_rows.append({"condition": c, "comparisons_unreported": len(hit), "reports": len({rid for rid, _ in hit}),
                         "report_ids": ";".join(sorted({rid for rid, _ in hit})),
                         "comparison_uids": ";".join(f"{rid}:{k['id']}" for rid, k in hit)})
    T["reporting_gaps"] = gap_rows
    st = stress_tests(records, meths)
    T["stress_tests"] = st
    scs = scs_pairs(records, grade)
    T["scs"] = scs
    used = []
    for s in scs:
        for u in s["result_uids"].split(";"):
            rid, xid = u.split(":", 1)
            r = next(x for x in records[rid]["results"] if x["id"] == xid)
            d = decisive.get(u, {})
            used.append({"result_uid": u, "used_in": "suppression cost " + s["comparison_uid"], "value": r["value"], "page": r["page"],
                         "location": r.get("location", ""), "code_check_04": d.get("code_check", "not a decisive value"),
                         "read_against_page": ("read by eye: matches" if read.get(u) == r["value"] else
                                               "NOT READ" if u not in read else f"read by eye: page shows {read[u]}")})
    T["values_used_05"] = used
    bad = [u for u in used if u["read_against_page"] != "read by eye: matches"]
    if bad:
        raise ValueError(f"a value the findings use was not read against its page or differs: {bad}")
    ids = _ids
    F = ["# Findings of notebook 05", "",
         f"Built {today} from the records committed with notebook 04 ({c04}) and later record patches. Every number names the file and the rows it was counted from.", ""]
    F += ["## 1. The map (RQ1)", "",
          f"- {len(meths)} methods in {len({m['report'] for m in meths})} reports: {sum(m['trained'] == 'yes' for m in meths)} trained, "
          f"{sum(m['trained'] == 'no' for m in meths)} prompting or fixed rules (analysis/methods.csv).",
          "- Methods per axis-1 value: " + "; ".join(f"{a} {by_a1[a]}" for a in AXIS1) + " (analysis/methods.csv, column axis1; a method with two values counts twice).",
          "- Methods per axis-2 value: " + "; ".join(f"{b} {by_a2[b]}" for b in AXIS2) + " (column axis2).",
          f"- {len(filled)} of {len(mp)} cells of the map hold at least one method; {len(mp) - len(filled)} are empty (analysis/map.csv, column methods).",
          "- The fullest cells: " + "; ".join(f"{r['axis1']} x {r['axis2']} {r['methods']}" for r in sorted(filled, key=lambda r: -r["methods"])[:6]) + " (analysis/map.csv).",
          f"- Methods with released weights: {sum(m['released_weights'] == 'yes' for m in meths)} (analysis/methods.csv, released_weights = yes: "
          f"{ids(m['method_uid'] for m in meths if m['released_weights'] == 'yes')}).",
          "- Methods per year of the report: " + "; ".join(f"{y} {n}" for y, n in sorted(Counter(m['year'] for m in meths).items())) + ".", ""]
    F += ["## 2. Signal dependence and layout compatibility (RQ1)", "",
          f"- {len(with_signal)} methods show a source signal at inference; {len(meths) - len(with_signal)} show none (axis 1 no_signal).",
          "- Trained methods with a signal: " + "; ".join(f"{k} {v}" for k, v in sig_t.most_common()) + ".",
          "- Prompting or fixed-rule methods with a signal: " + "; ".join(f"{k} {v}" for k, v in sig_p.most_common()) + ".",
          "- Run with the signal removed or moved (measured or measured_other_checkpoint): "
          + ids([m["method_uid"] for m in with_signal if m["signal_dependence"].startswith("measured")], 20)
          + " (analysis/signal_dependence.csv, with what was found in signal_dependence_detail).",
          *[f"- Layout {x}: " + "; ".join(f"{k} {v}" for k, v in Counter(m[f'layout_{x}'] for m in meths).most_common()) + " (analysis/layout_compat.csv)."
            for x in LAYOUTS], ""]
    F += ["## 3. Coverage of the three jobs (RQ2)", "",
          *[f"- {q}: papers reporting it directly {sum(p[q] == 'direct' for p in paper_rows)}, by proxy {sum(p[q] == 'proxy' for p in paper_rows)}, "
            f"only pooled {sum(p[q] == 'aggregate_only' for p in paper_rows)}; evaluated configurations covering it {sum(c[q] != 'none' for c in cfg)} of {len(cfg)} "
            f"(analysis/coverage_by_paper.csv and coverage_by_configuration.csv)." for q in QUANTITIES],
          f"- All three jobs scored on the same replies: {pct(sum(c['joint_all_three'] == 'yes' for c in cfg), len(cfg))} configurations, in "
          f"{len({c['report'] for c in cfg if c['joint_all_three'] == 'yes'})} reports: {ids({c['report'] for c in cfg if c['joint_all_three'] == 'yes'})} "
          "(coverage_by_configuration.csv, joint_all_three = yes).",
          f"- R1 and R2attack on the same replies: {pct(sum(c['joint_R1_R2attack'] == 'yes' for c in cfg), len(cfg))} configurations.",
          "- Own paper against anyone, for the named families proposed in the corpus (analysis/coverage_by_family.csv):",
          *[f"  - {f['family_key']}: R3 measured by its own paper: {'yes' if any(f[q + '_original'] == 'yes' for q in ('R3ns', 'R3cp', 'R3safe')) else 'no'}; "
            f"by anyone: {'yes' if any(f[q + '_anyone'] == 'yes' for q in ('R3ns', 'R3cp', 'R3safe')) else 'no'}; later reports: "
            f"{len([x for x in f['reports_measuring_it_later'].split(';') if x])}" for f in fam], ""]
    r1 = [r for r in cw if r["quantity"] == "R1"]
    r1n = sum(r["results"] for r in r1)
    F += ["## 4. What the metrics count (RQ2, the crosswalk)", "",
          f"- {len(cw)} benchmark, quantity and judge-rule groups over {sum(r['results'] for r in cw)} results (analysis/crosswalk.csv).",
          *[f"- R1 results whose judge rule is {j}: {pct(sum(r['results'] for r in r1 if r['judge_rule'] == j), r1n)}; it cannot see: "
            f"{next(r['cannot_see'] for r in r1 if r['judge_rule'] == j)}." for j in sorted({r['judge_rule'] for r in r1})],
          f"- Results on general-utility benchmarks (not the user's task on the document): {sum(r['results'] for r in cw if r['quantity'] == 'general_utility')}.", ""]
    sec = [r for r in cs if "R1" in r["jobs"].split(";")]
    utl = [r for r in cs if r["utility_basis"]]
    xs = [r for r in od if r["purpose"] == "X"]
    F += ["## 5. Support for headline claims (RQ3)", "",
          f"- {len(cs)} headline claims in {len({r['report'] for r in cs})} reports; best grade of their supporting comparisons: "
          + "; ".join(f"{g} {sum(r['best_grade'] == g for r in cs)}" for g in ["A", "B", "C", "U", "none"]) + " (analysis/claim_support.csv).",
          f"- Claims about security (job R1) supported by at least one grade-A comparison: {pct(sum(r['best_grade'] == 'A' for r in sec), len(sec))}.",
          f"- Claims about utility that rest on general-utility metrics only: {pct(sum(r['utility_basis'] == 'general utility only' for r in utl), len(utl))} "
          f"({ids(r['claim_uid'] for r in utl if r['utility_basis'] == 'general utility only')}).",
          f"- Claims with a result in the same report that cuts against them: {sum(r['contradicted_by'] > 0 for r in cs)} "
          f"({ids(r['claim_uid'] for r in cs if r['contradicted_by'] > 0)}).",
          f"- Cross-paper readings (purpose X): {len(xs)}, of which grade U: {sum(r['grade'] == 'U' for r in xs)} (analysis/orderings.csv, purpose = X). "
          "An unreported condition on either side leaves the reading not assessable.",
          "- Conditions left unreported in comparisons: " + "; ".join(f"{g['condition']} {g['comparisons_unreported']} (reports {g['reports']})" for g in gap_rows)
          + " (analysis/reporting_gaps.csv); these are the lines the Injection Defence Card asks for.", ""]
    wa = [r for r in od if r["grade"] == "A" and r["purpose"] in ("W", "M")]
    F += ["## 6. Orderings", "",
          f"- Grade-A comparisons between two defended arms inside one report: {len(wa)} ({sum(r['purpose'] == 'W' for r in wa)} whole-configuration, "
          f"{sum(r['purpose'] == 'M' for r in wa)} mechanism) in {len({r['report'] for r in wa})} reports (analysis/orderings.csv, grade = A). "
          "These are the only orderings the review states, each inside its own report.",
          f"- No ordering across papers is supported: every cross-paper reading is grade {', '.join(sorted({r['grade'] for r in xs})) or 'none'}.", ""]
    F += ["## 7. Stress tests (RQ4)", "",
          f"- Methods that faced an adaptive, optimisation, RL or human attacker, in their own paper or a later one: {pct(sum(r['pushed_hard'] == 'yes' for r in st), len(st))} "
          "(analysis/stress_tests.csv).",
          f"- Faced one in their own paper: {sum(bool(r['own_stress_attackers']) for r in st)}; only in a later paper: "
          f"{sum(bool(r['later_stress_attackers']) and not r['own_stress_attackers'] for r in st)}; white-box attack in their own paper: {sum(r['own_white_box'] > 0 for r in st)}.",
          f"- Run with the signal removed or moved: {sum(r['signal_removal_run'] == 'yes' for r in st)}; re-measured by a later paper: "
          f"{sum(bool(r['later_reports']) for r in st)}.",
          "- Trained methods pushed by a stronger attacker: " + pct(sum(r['pushed_hard'] == 'yes' for r in st if r['trained'] == 'yes'), sum(r['trained'] == 'yes' for r in st))
          + "; prompting or fixed-rule methods: " + pct(sum(r['pushed_hard'] == 'yes' for r in st if r['trained'] == 'no'), sum(r['trained'] == 'no' for r in st)) + ".", ""]
    F += ["## 8. Suppression cost per security point", "",
          f"- Published pairs with security and non-suppression for a defended and the undefended configuration in one grade-A comparison: {len(scs)}, "
          f"all in {ids({s['report'] for s in scs})} (analysis/scs.csv).",
          *[f"- {s['defence']} on {s['base']}: SCS {s['SCS']} (R3ns {s['R3ns_undefended']} to {s['R3ns_defended']}, R1 {s['R1_undefended']} to {s['R1_defended']}; "
            f"{s['result_uids']})." for s in scs],
          f"- The {len({u['result_uid'] for u in used})} printed values behind these pairs were read by eye against their page (analysis/values_used_05.csv). "
          "SCS is computed only from published pairs and is not called a benchmark.", ""]
    D = [f"decision_05: ANALYSIS DONE; GATE G4 PASSED", f"this step: {today}",
         f"from the records of notebook 04 ({c04}) and later record patches; outputs in reports/analysis/ and reports/findings_05.md",
         f"methods {len(meths)}; cells holding a method {len(filled)} of {len(mp)}; claims {len(cs)}; configurations {len(cfg)}",
         f"supported orderings: {len(wa)} grade-A comparisons between two defended arms, each inside its own report ({len({r['report'] for r in wa})} reports), "
         "listed in reports/analysis/orderings.csv",
         f"not supported: any ordering across papers ({len(xs)} cross-paper readings, grade {', '.join(sorted({r['grade'] for r in xs})) or 'none'})",
         f"printed values the findings use: {len({u['result_uid'] for u in used})}, every one read by eye against its page",
         "next: notebook 06_figures (figures 1 to 10 at 600 dpi with paper names; dpi, size and label-overlap checks)"]
    return {"tables": T, "findings": F, "decision": D,
            "summary": {"methods": len(meths), "cells": len(filled), "claims": len(cs), "configurations": len(cfg), "orderings_A": len(wa),
                        "cross_paper": len(xs), "scs": len(scs)}}

In [ ]:
%%writefile tests/test_analysis.py
"""Tests for src/analysis.py on a two-report fixture: a proposing report and a later re-measurement."""
import importlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "src"))
A = importlib.import_module("analysis")
schema_mod = importlib.import_module("schema")

M = {c: "matched" for c in schema_mod.CONDITIONS}


def _ev(i, did, attacker="static", access="black_box"):
    return {"id": i, "defence_id": did, "benchmark": "SEP test", "attacker_type": attacker, "attacker_access": access,
            "judge": "substring match on the payload", "layout_same_as_defence": "yes", "measured_by": "x", "page": 3}


def _res(i, ev, q, v, direction="lower_better", unit="percent", also=None):
    r = {"id": i, "evaluation_id": ev, "quantity": q, "coverage": "direct", "metric_as_printed": q + " rate", "value": v,
         "unit": unit, "direction": direction, "qualifier": "exact", "value_source": "table", "page": 3}
    if also:
        r["also_quantity"] = also
    return r


def fixture():
    a = {"report": {"id": "a", "slug": "a", "role": "proposes", "read": "F", "first_date": "2025-01-01"},
         "coverage": {q: "not_reported" for q in A.QUANTITIES} | {"R1": "direct", "R3ns": "direct"},
         "defences": [{"id": "sa", "name": "SecAlign", "origin": "this_report", "base_checkpoint": "M", "axis1": ["delimiters_markers"],
                       "axis2": ["preference_optimisation"], "isolation": "ablation_present", "signal_dependence": "measured",
                       "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2,
                       "layout_compat": {"user_turn": "supported", "separate_role": "untested"}},
                      {"id": "abl", "name": "SecAlign without delimiters", "origin": "this_report", "base_checkpoint": "M",
                       "axis1": ["no_signal"], "axis2": ["preference_optimisation"], "isolation": "not_applicable",
                       "signal_dependence": "not_applicable", "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2},
                      {"id": "sw", "name": "Sandwich defence", "origin": "other_report", "base_checkpoint": "M", "axis1": ["position"],
                       "axis2": ["prompting"], "isolation": "not_applicable", "signal_dependence": "baseline_only",
                       "trains_instruction_pair": "not_applicable", "trains_carry_through": "not_applicable", "page": 2}],
         "evaluations": [_ev("e0", "none:M"), _ev("e1", "sa"), _ev("e2", "sw"), _ev("e3", "sa", "adaptive", "white_box")],
         "results": [_res("r0", "e0", "R1", "50.0"), _res("i0", "e0", "R3ns", "10.0"),
                     _res("r1", "e1", "R1", "10.0"), _res("i1", "e1", "R3ns", "30.0"), _res("u1", "e1", "R2attack", "60.0", "higher_better"),
                     _res("r2", "e2", "R1", "20.0"), _res("r3", "e3", "R1", "40.0")],
         "comparisons": [{"id": "c1", "left": "e1", "right": "e0", "purpose": "W", "varies": "defence", **M, "page": 3},
                         {"id": "c2", "left": "e1", "right": "e2", "purpose": "W", "varies": "defence", **M, "page": 3},
                         {"id": "c3", "left": "e1", "right": "e0", "purpose": "W", "varies": "defence", **M | {"c5_judge": "unreported"}, "page": 3}],
         "claims": [{"id": "k1", "text": "SecAlign stops attacks", "location": "abstract", "anchor": "a", "jobs": ["R1"],
                     "supported_by": ["c1"], "page": 1},
                    {"id": "k2", "text": "utility is kept", "location": "abstract", "anchor": "b", "jobs": ["general_utility"],
                     "supported_by": ["c3"], "page": 1}]}
    b = {"report": {"id": "b", "slug": "b", "role": "re_measures", "read": "F", "first_date": "2026-01-01"},
         "coverage": {q: "not_reported" for q in A.QUANTITIES} | {"R3cp": "direct"},
         "defences": [{"id": "s2", "name": "SecAlign (re-measured)", "origin": "other_report", "base_checkpoint": "M",
                       "axis1": ["delimiters_markers"], "axis2": ["preference_optimisation"], "isolation": "not_applicable",
                       "signal_dependence": "not_reported", "trains_instruction_pair": "no", "trains_carry_through": "no", "page": 2}],
         "evaluations": [_ev("f1", "s2", "optimisation")],
         "results": [_res("x1", "f1", "R3cp", "70.0", "higher_better"), _res("x2", "f1", "R1", "30.0"),
                     _res("x3", "f1", "R2attack", "50.0", "higher_better", also=["R3safe"])]}
    return {"a": a, "b": b}


def test_methods_map_and_families():
    recs = fixture()
    ms = A.methods(recs)
    assert [m["name"] for m in ms] == ["SecAlign"]                       # the variant and the other report's baseline are not methods
    m = ms[0]
    assert m["layout_user_turn"] == "supported" and m["layout_system_context"] == "untested" and m["signal_dependence"] == "measured"
    cells = {(r["axis1"], r["axis2"]): r for r in A.map_table(ms)}
    assert cells[("delimiters_markers", "preference_optimisation")]["methods"] == 1 and cells[("no_signal", "sft")]["methods"] == 0
    assert A.family("Meta-SecAlign-70B + EP-DPO") is None and A.family("SecAlign++ LoRA") == "Meta-SecAlign"
    assert A.family("repeat the user's prompt").startswith("Sandwich") and A.family("White-box defence: reminder") is None


def test_coverage_original_and_by_anyone_and_joint():
    recs = fixture()
    cfg = A.configuration_coverage(recs)
    by = {r["defence_uid"]: r for r in cfg}
    assert by["a:sa"]["R1"] == "direct" and by["a:sa"]["R3ns"] == "direct" and by["a:sa"]["joint_all_three"] == "yes"
    assert by["b:s2"]["R3safe"] == "proxy" and by["b:s2"]["joint_all_three"] == "yes"
    fam = {r["family_key"]: r for r in A.family_coverage(recs, cfg, A.methods(recs))}
    f = fam["SecAlign | trained"]
    assert f["R3cp_original"] == "no" and f["R3cp_anyone"] == "yes" and f["reports_measuring_it_later"] == "b"


def test_crosswalk_claims_orderings_stress_and_scs():
    recs = fixture()
    cw = A.crosswalk(recs)
    r1 = next(r for r in cw if r["quantity"] == "R1")
    assert r1["judge_rule"] == "string or answer match" and "Rp" in r1["cannot_see"]
    cs = {r["claim_uid"]: r for r in A.claim_support(recs, schema_mod.grade)}
    assert cs["a:k1"]["best_grade"] == "A" and cs["a:k2"]["best_grade"] == "U"
    assert cs["a:k2"]["utility_basis"] == "R2 results"
    od = A.orderings(recs, schema_mod.grade)
    assert [o["comparison_uid"] for o in od] == ["a:c2"] and od[0]["paired_results"] == 1
    st = {r["method_uid"]: r for r in A.stress_tests(recs, A.methods(recs))}
    s = st["a:secalign"]
    assert s["own_stress_attackers"] == "adaptive" and s["own_white_box"] == 1 and s["later_stress_attackers"] == "optimisation"
    sc = A.scs_pairs(recs, schema_mod.grade)
    assert len(sc) == 1 and sc[0]["SCS"] == 0.5                         # (90 - 70) / (90 - 50)


def test_run_writes_every_table_and_checks_read_values():
    recs = fixture()
    import pytest
    with pytest.raises(ValueError):
        A.run(recs, schema_mod.grade, schema_mod.CONDITIONS, "abc1234", {}, {}, "2026-01-01")
    read = {"a:r1": "10.0", "a:r0": "50.0", "a:i1": "30.0", "a:i0": "10.0"}
    out = A.run(recs, schema_mod.grade, schema_mod.CONDITIONS, "abc1234", read, {}, "2026-01-01")
    assert set(out["tables"]) == {"methods", "map", "signal_dependence", "layout_compat", "coverage_by_paper", "coverage_by_configuration",
                                  "coverage_by_family", "crosswalk", "claim_support", "orderings", "reporting_gaps", "stress_tests", "scs", "values_used_05"}
    assert out["summary"]["methods"] == 1 and out["summary"]["scs"] == 1
    assert any("SCS 0.5" in line for line in out["findings"]) and out["decision"][0].startswith("decision_05")

In [ ]:
import schema, search, corpus, paths, extract, analysis
for m in (schema, search, corpus, paths, extract, analysis):
    importlib.reload(m)
t = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_analysis.py", "tests/test_extract.py"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout.strip() else t.stderr.strip()[-400:])

## Stage 3. The patch: comparison `c_ise` in the SecFid record

In [ ]:
import copy, yaml, hashlib, datetime as dt
from collections import Counter
RID = "secfid_arxiv2026"
p = paths.RECORDS / f"{RID}.yaml"
text = p.read_text(encoding="utf-8")
rec = yaml.safe_load(text)
dump = lambda r: yaml.safe_dump(r, sort_keys=False, width=220, allow_unicode=True)
if dump(rec) != text:
    raise SystemExit("STOP: the record does not round-trip byte for byte; the patch would rewrite unrelated lines")
comps = {c["id"]: c for c in rec["comparisons"]}
evs = {e["id"]: e for e in rec["evaluations"]}
if "c_ise" in comps:
    raise SystemExit("STOP: c_ise already exists; the patch was applied before")
for need in ("ev_ise", "ev_base", "c_aside"):
    if need not in comps and need not in evs:
        raise SystemExit(f"STOP: {need} not in the record")
new = copy.deepcopy(comps["c_aside"])
new["id"], new["left"], new["right"] = "c_ise", "ev_ise", "ev_base"
before = copy.deepcopy(rec)
rec["comparisons"].insert(list(comps).index("c_aside") + 1, new)
k1 = next(k for k in rec["claims"] if k["id"] == "k1")
k1["supported_by"] = [x for x in k1["supported_by"] if x != "c_ise"] + ["c_ise"]
changed = [("comparisons", "c_ise", "added: ev_ise against ev_base, purpose W, conditions as c_aside, page 6"),
           ("claims", "k1", f"supported_by: {before['claims'][0]['supported_by']} -> {k1['supported_by']}")]
for kind, uid, what in changed:
    print(f"  {kind}:{uid}: {what}")
print("comparison added:", {k: v for k, v in new.items()})

## Stage 4. Checks on the patched record: schema, PDF, amended rules, corpus

In [ ]:
import re, logging
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader
import pypdfium2 as pdfium
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
slug = rec["report"]["slug"]
pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
if not pdf.exists() or corpus.sha256(pdf) != man[slug]["sha256"]:
    raise SystemExit("STOP: the SecFid PDF is missing or differs from the registered one")
rd = PdfReader(str(pdf))
plain = [pg.extract_text() or "" for pg in rd.pages]
lay = [pg.extract_text(extraction_mode="layout") or "" for pg in rd.pages]
doc = pdfium.PdfDocument(str(pdf))
def third(page):
    return doc[page - 1].get_textpage().get_text_range()
probs, notes = schema.validate_record(rec, S1, plain, lay)
kept = []
for x in probs:
    m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", x)
    if not (m and schema.value_on_page(m.group(2), third(int(m.group(3))))):
        kept.append(x)
probs = kept + extract.rule_problems(rec)
recs = {f.stem: schema.load_record(f) for f in sorted(paths.RECORDS.glob("*.yaml"))}
recs[RID] = rec
included = sorted(s for s, m in man.items() if m.get("decision") == "include")
probs += extract.cross_record_problems(recs, included, {RID: len(rd.pages)})
for x in probs:
    print("PROBLEM:", x)
if probs:
    raise SystemExit("STOP: the patched record does not validate; nothing was written")
grades = Counter(schema.grade(c) for c in rec["comparisons"])
print(f"patched record validates; comparisons {len(rec['comparisons'])}, grades {dict(grades)}; c_ise grade {schema.grade(new)}")
p.write_text(dump(rec), encoding="utf-8")
print("written:", p)

## Stage 5. The extraction tables and the decisive values, regenerated (notebook 04's code paths)

In [ ]:
out_dir = paths.REPORTS / "extraction"
tables = extract.build_tables(recs, S1, schema.grade)
for name, (rows, fields) in tables.items():
    search.write_csv(out_dir / f"{name}.csv", rows, fields)
print("tables:", ", ".join(f"{n} {len(t[0])}" for n, t in tables.items()))
old_dec = search.read_csv(out_dir / "decisive_values_04.csv")
keep = [r for r in old_dec if r["report"] != RID]
evsd = {e["id"]: e for e in rec["evaluations"]}
names = {d["id"]: d["name"] for d in rec["defences"]}
new_rows = []
for xid, why in sorted(extract.decisive_results(rec, schema.grade).items()):
    r = next(x for x in rec["results"] if x["id"] == xid)
    e = evsd.get(r["evaluation_id"], {})
    pg = int(r["page"])
    n_plain = extract.value_count(r["value"], plain[pg - 1], schema.norm_value)
    n_lay = extract.value_count(r["value"], lay[pg - 1], schema.norm_value)
    if n_plain or n_lay:
        found, n = "pypdf", max(n_plain, n_lay)
    else:
        found, n = "pypdfium2 (third reading)", extract.value_count(r["value"], third(pg), schema.norm_value)
    if n == 0:
        raise SystemExit(f"STOP: decisive value {RID}:{xid} not found on page {pg}")
    check = ("one occurrence on the page" if n == 1 and found == "pypdf" else
             "more than one occurrence on the page: read the cell" if found == "pypdf" else "third reading only: read the cell")
    did = e.get("defence_id", "")
    new_rows.append({"result_uid": f"{RID}:{xid}", "report": RID, "page": pg, "location": r.get("location", ""),
                     "arm": "undefended (" + did[5:] + ")" if str(did).startswith("none:") else names.get(did, did),
                     "benchmark": e.get("benchmark", ""), "quantity": r["quantity"], "metric_as_printed": r["metric_as_printed"],
                     "value": r["value"], "unit": r.get("unit", ""), "value_source": r.get("value_source", ""),
                     "why_decisive": "; ".join(sorted(why)), "found_by": found, "occurrences_on_page": n, "code_check": check})
dec_rows = sorted(keep + new_rows, key=lambda r: r["result_uid"])
search.write_csv(out_dir / "decisive_values_04.csv", dec_rows, list(old_dec[0].keys()))
old_secfid = [r for r in old_dec if r["report"] == RID]
print(f"decisive values: {len(old_dec)} -> {len(dec_rows)} (SecFid {len(old_secfid)} -> {len(new_rows)}); other reports' rows unchanged")
summ = search.read_csv(out_dir / "summary_04.csv")
for s in summ:
    if s["report"] == RID:
        s["comparisons"] = str(len(rec["comparisons"]))
search.write_csv(out_dir / "summary_04.csv", summ, list(summ[0].keys()))
doc.close()

## Stage 6. The analysis outputs, regenerated (notebook 05's output stage, now `analysis.run`)

In [ ]:
READ = {'secfid_arxiv2026:r_base_e': '49.1', 'secfid_arxiv2026:r_base_i': '7.7', 'secfid_arxiv2026:r_aside_e': '5.7', 'secfid_arxiv2026:r_aside_i': '24.7', 'secfid_arxiv2026:r_msa_e': '0.7', 'secfid_arxiv2026:r_msa_i': '26.1', 'secfid_arxiv2026:r_dt_e': '1.9', 'secfid_arxiv2026:r_dt_i': '53.7', 'secfid_arxiv2026:r_ise_e': '14.0', 'secfid_arxiv2026:r_ise_i': '20.3'}
dec = {r["result_uid"]: r for r in dec_rows}
out = analysis.run(recs, schema.grade, schema.CONDITIONS, C04, READ, dec, dt.date.today().isoformat())
A = paths.REPORTS / "analysis"
for name, rows in out["tables"].items():
    search.write_csv(A / f"{name}.csv", rows, list(rows[0].keys()))
(paths.REPORTS / "findings_05.md").write_text("\n".join(out["findings"]) + "\n")
(paths.REPORTS / "decision_05.txt").write_text("\n".join(out["decision"]) + "\n")
print("\n".join(l for l in out["findings"] if l.startswith("## 8") or "SCS" in l or "inputs were read" in l))
print("summary:", out["summary"])

## Stage 7. Decision record and tests

In [ ]:
scs = out["tables"]["scs"]
lines = [f"decision_05b: SECFID RECORD PATCHED (c_ise added); EXTRACTION AND ANALYSIS OUTPUTS REGENERATED",
         f"this step: {dt.date.today().isoformat()}",
         "patch: reports/records/secfid_arxiv2026.yaml, comparison c_ise (ev_ise against ev_base, purpose W, conditions as c_aside) and c_ise added to claim k1",
         f"checks: schema, every value and anchor on its page, amended rules, corpus-wide; comparisons in the record {len(rec['comparisons'])}, all grade A",
         f"regenerated from the records: reports/extraction/*.csv ({sum(len(t[0]) for t in tables.values())} rows; comparisons {len(tables['comparisons'][0])}), "
         f"reports/extraction/decisive_values_04.csv ({len(dec_rows)} rows), reports/analysis/*.csv, reports/findings_05.md, reports/decision_05.txt",
         f"decision_04.txt is left as written; its counts predate this patch: comparisons {len(tables['comparisons'][0]) - 1} -> {len(tables['comparisons'][0])}, "
         f"decisive values {len(old_dec)} -> {len(dec_rows)}",
         f"suppression cost pairs: {len(scs)}: " + "; ".join(f"{s['defence'].split(' (')[0]} {s['SCS']}" for s in scs),
         f"printed values the findings use: {len({u['result_uid'] for u in out['tables']['values_used_05']})}, every one read by eye against its page",
         "next: notebook 06_figures"]
(paths.REPORTS / "decision_05b.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 8. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "05b SecFid record patch: the ISE arm of Table 1 gets its comparison; extraction tables, decisive values, analysis outputs and findings regenerated from the records"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)